In [1]:
### VERY IMPORTANT!!!
# create a .venv and install dependencies
# Run in your terminal: "python -m pip install pandas ipykernel numpy jupyter"

In [2]:
# import sys
# print(sys.executable)

In [3]:
import pandas as pd
import numpy as np

In [4]:

gravity_base = pd.read_csv("../outputs_csv/gravity_base.csv")



/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_92456/691074815.py:1: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  gravity_base = pd.read_csv("../outputs_csv/gravity_base.csv")


In [5]:
gravity_base['dropBackType'].value_counts()

dropBackType
Traditional               5645596
Scramble                  1180300
Unknown                    532224
Designed Rollout Right     259886
Scramble Rollout Right     159126
Designed Rollout Left      142736
Scramble Rollout Left       28578
Designed Run                 4246
Name: count, dtype: int64

All plays with dropBackType = "Unknown" are plays that were nullified by penalty. These plays can be dropped. Screens and RPOs were automatically dropped by Kaggle.

In [6]:
# exclude rollouts, designed runs, and plays nullified by penalty (Unknown)
base_filtered = gravity_base[(gravity_base['dropBackType'] == 'Traditional') | (gravity_base['dropBackType'] == 'Scramble')]

In [7]:
each_play = base_filtered.dropna(subset='event')
each_play=each_play.drop_duplicates(subset=['gameId', 'playId', 'frameID'])
each_play['event'].value_counts()

event
ball_snap                    7417
pass_forward                 6570
autoevent_ballsnap           3279
autoevent_passforward        3250
play_action                  1351
qb_sack                       412
run                           378
pass_arrived                  319
autoevent_passinterrupted     166
man_in_motion                 138
line_set                      121
shift                         101
pass_tipped                    96
first_contact                  68
qb_strip_sack                  57
pass_outcome_incomplete        34
pass_outcome_caught            20
fumble                         12
handoff                        10
fumble_offense_recovered        7
huddle_break_offense            2
tackle                          2
dropped_pass                    1
penalty_flag                    1
Name: count, dtype: int64

In [8]:
each_play = base_filtered.dropna(subset='event')
each_play=each_play.drop_duplicates(subset=['gameId', 'playId', 'frameID'])

# get the frames where the window ends. Window ends when a forward pass is thrown, 
# or when a QB gets sacked, or when a QB gets strip-sacked, or when a QB takes off to run.
end_window_event = each_play[(each_play['event'] == 'pass_forward') | (each_play['event'] == 'qb_sack')
| (each_play['event'] == 'qb_strip_sack') | (each_play['event'] == 'run')][['gameId', 'playId', 'frameID']]
end_window_event = end_window_event.sort_values(by=['gameId', 'playId', 'frameID'])
end_window_event = end_window_event.drop_duplicates(subset=['gameId', 'playId'], keep='first')

# get the frames where the ball is snapped
ball_snaps = each_play[each_play['event'] == 'ball_snap'][['gameId', 'playId','frameID']]
ball_snaps = ball_snaps.drop_duplicates(subset=['gameId', 'playId'], keep='first')

# get all plays where a ball_snapped frame and an end_window frame exists
each_play=each_play.merge(end_window_event[['gameId','playId']], on=['gameId','playId'],how='inner')
each_play=each_play.merge(ball_snaps[['gameId','playId']], on=['gameId','playId'],how='inner')
merged_temp = pd.concat([end_window_event, ball_snaps])
each_play=each_play.merge(merged_temp[['gameId','playId','frameID']], on=['gameId','playId','frameID'], how='inner')

In [9]:
# confirm that each play has two frames: a ball_snap frame and an end_window frame
value_counts = each_play.groupby(['gameId','playId'])['event'].count()
value_counts.value_counts()


event
2    7386
Name: count, dtype: int64

In [10]:
temp = base_filtered.merge(each_play[['gameId','playId']].drop_duplicates(), on=['gameId','playId'], how='inner') 
ball_snaps = each_play[each_play['event'] == 'ball_snap'] 
ball_snaps['frameIdBallSnap'] = ball_snaps['frameID'] 
end_window_event = each_play[each_play['event'] != 'ball_snap'] 
end_window_event['frameIdEndWindow'] = end_window_event['frameID'] 
temp = temp.merge(ball_snaps[['gameId','playId','frameIdBallSnap']], on=['gameId','playId'], how='inner') 
temp = temp.merge(end_window_event[['gameId','playId','frameIdEndWindow']], on=['gameId','playId'], how='inner')

In [11]:
temp["first_frame_tt_exceeds_3_0"] = temp["frameIdBallSnap"] + 31

# --------------------------------------------------
# 1. Column definitions
# --------------------------------------------------
GAME_COL = "gameId"
PLAY_COL = "playId"
FRAME_COL = "frameID"
EVENT_COL = "event"
POSITION_COL = "pff_positionLinedUp"
Y_COL = "y"

# Work on a copy
temp = temp.copy()

# --------------------------------------------------
# 2. Get LT y-position at ball_snap for each play
# --------------------------------------------------
lt_snap = (
    temp.loc[
        (temp[EVENT_COL] == "ball_snap") &
        (temp[POSITION_COL] == "LT"),
        [GAME_COL, PLAY_COL, Y_COL]
    ]
    .drop_duplicates(subset=[GAME_COL, PLAY_COL])
    .rename(columns={Y_COL: "y_LT"})
)

# --------------------------------------------------
# 3. Get RT y-position at ball_snap for each play
# --------------------------------------------------
rt_snap = (
    temp.loc[
        (temp[EVENT_COL] == "ball_snap") &
        (temp[POSITION_COL] == "RT"),
        [GAME_COL, PLAY_COL, Y_COL]
    ]
    .drop_duplicates(subset=[GAME_COL, PLAY_COL])
    .rename(columns={Y_COL: "y_RT"})
)

# --------------------------------------------------
# 4. Merge y_LT and y_RT onto the tracking dataframe
#    Inner join requires both LT and RT to exist for the play
# --------------------------------------------------
df_with_tackles = (
    temp.merge(lt_snap, on=[GAME_COL, PLAY_COL], how="inner", validate="many_to_one")
        .merge(rt_snap, on=[GAME_COL, PLAY_COL], how="inner", validate="many_to_one")
)

# --------------------------------------------------
# 5. Identify QB rows
# --------------------------------------------------
qb_tracking = df_with_tackles.loc[
    df_with_tackles[POSITION_COL].eq("QB")
].copy()

# --------------------------------------------------
# 6. Define tackle box boundaries using snap-time tackle locations
# --------------------------------------------------
qb_tracking["tackle_box_min_y"] = qb_tracking[["y_LT", "y_RT"]].min(axis=1)
qb_tracking["tackle_box_max_y"] = qb_tracking[["y_LT", "y_RT"]].max(axis=1)

# --------------------------------------------------
# 7. Flag whether QB is outside tackle box on each frame
# --------------------------------------------------
qb_tracking["qb_outside_tackle_box"] = (
    (qb_tracking[Y_COL] < qb_tracking["tackle_box_min_y"]) |
    (qb_tracking[Y_COL] > qb_tracking["tackle_box_max_y"])
)

# --------------------------------------------------
# 8. Find the FIRST frame where QB is outside tackle box for each play
#    This is an EXCLUSIVE upper-bound candidate
# --------------------------------------------------
first_qb_outside = (
    qb_tracking.loc[qb_tracking["qb_outside_tackle_box"], [GAME_COL, PLAY_COL, FRAME_COL]]
    .groupby([GAME_COL, PLAY_COL], as_index=False)[FRAME_COL]
    .min()
    .rename(columns={FRAME_COL: "frameIdFirstOutsideTackleBox"})
)

# --------------------------------------------------
# 9. Merge that first-outside frame back to all rows
# --------------------------------------------------
df_with_tackles = df_with_tackles.merge(
    first_qb_outside,
    on=[GAME_COL, PLAY_COL],
    how="left",
    validate="many_to_one"
)

# --------------------------------------------------
# 10. Build the EXCLUSIVE upper bound
#
# Existing frameIdEndWindow is assumed to be the first event frame
# (pass_forward / qb_sack / qb_strip_sack / etc.), which should be excluded.
#
# Time-to-throw > 3.0 sec means the first excluded frame is ball_snap + 31.
# QB outside tackle box also excludes that first outside frame itself.
# --------------------------------------------------
exclusive_bound_candidates = [
    "frameIdEndWindow",
    "first_frame_tt_exceeds_3_0",
    "frameIdFirstOutsideTackleBox"
]

df_with_tackles["frameIdEndWindowExclusive"] = df_with_tackles[exclusive_bound_candidates].min(axis=1)

# --------------------------------------------------
# 11. Keep only frames STRICTLY BEFORE the exclusive upper bound
# --------------------------------------------------
tracking_only_when_qb_in_tackle_box = df_with_tackles.loc[
    df_with_tackles[FRAME_COL] < df_with_tackles["frameIdEndWindowExclusive"]
].copy()

# --------------------------------------------------
# 12. Apply lower bound: keep frames at least 5 after ball snap
# --------------------------------------------------
filtered_lower_and_upper_bound = tracking_only_when_qb_in_tackle_box.loc[
    tracking_only_when_qb_in_tackle_box[FRAME_COL] >= tracking_only_when_qb_in_tackle_box["frameIdBallSnap"] + 5
].copy()

# --------------------------------------------------
# 13. Convert back to an INCLUSIVE frameIdEndWindow for downstream cells
#     The next cell expects frameIdEndWindow to be the last included frame.
# --------------------------------------------------
max_upper_bound = (
    filtered_lower_and_upper_bound
    .groupby([GAME_COL, PLAY_COL])[FRAME_COL]
    .max()
    .reset_index()
    .rename(columns={FRAME_COL: "frameIdEndWindow"})
)

filtered_lower_and_upper_bound = filtered_lower_and_upper_bound.drop(
    columns=[
        "frameIdEndWindow",
        "frameIdEndWindowExclusive",
        "frameIdFirstOutsideTackleBox",
        "first_frame_tt_exceeds_3_0"
    ],
    errors="ignore"
)

filtered_lower_and_upper_bound = filtered_lower_and_upper_bound.merge(
    max_upper_bound,
    on=[GAME_COL, PLAY_COL],
    how="inner",
    validate="many_to_one"
)

In [12]:
# confirm that the longest frame window is 2.5 seconds long.

(filtered_lower_and_upper_bound.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - filtered_lower_and_upper_bound.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].max()

np.int64(25)

In [13]:
# filter out plays where the upper_bound - ball_snap <= 15

play_windows = (
    filtered_lower_and_upper_bound[['gameId', 'playId', 'frameIdBallSnap', 'frameIdEndWindow']]
    .drop_duplicates()
)

play_windows['window_length'] = (
    play_windows['frameIdEndWindow'] - play_windows['frameIdBallSnap']
)

valid_plays = play_windows[play_windows['window_length'] >= 15][['gameId', 'playId']]

tracking_filtered_by_window_length = filtered_lower_and_upper_bound.merge(
    valid_plays,
    on=['gameId', 'playId'],
    how='inner'
)
tracking_filtered_by_window_length

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,s,a,dis,o,dir,event,frameIdBallSnap,y_LT,y_RT,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.100,...,1.38,1.61,0.13,119.88,261.95,NaN,6,26.92,21.31,36
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.200,...,1.57,1.43,0.15,113.65,260.27,NaN,6,26.92,21.31,36
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.300,...,1.82,1.37,0.19,107.99,261.01,NaN,6,26.92,21.31,36
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.400,...,2.23,1.63,0.25,114.80,264.15,NaN,6,26.92,21.31,36
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.500,...,2.36,1.28,0.23,119.55,263.17,NaN,6,26.92,21.31,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3466007,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,7.47,0.28,0.74,48.35,60.83,NaN,7,26.85,20.70,37
3466008,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,7.45,0.67,0.75,48.35,61.74,NaN,7,26.85,20.70,37
3466009,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,7.48,1.43,0.75,58.99,63.79,NaN,7,26.85,20.70,37
3466010,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,7.41,2.01,0.75,63.66,65.69,NaN,7,26.85,20.70,37


In [14]:
# check if i filtered frames correctly


check_df = (
    tracking_filtered_by_window_length
    .groupby(['gameId', 'playId'], as_index=False)['frameID']
    .min()
    .merge(
        tracking_filtered_by_window_length[['gameId', 'playId', 'frameIdBallSnap']].drop_duplicates(),
        on=['gameId', 'playId'],
        how='left'
    )
)

check_df['matches'] = check_df['frameID'] == check_df['frameIdBallSnap'] + 5
print((check_df['matches'] == False).sum())

print((tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].min())

print((tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].max().reset_index() - tracking_filtered_by_window_length.groupby(['gameId', 'playId'])['frameID'].min().reset_index())['frameID'].max())


# check if QB is within tackle box at all times
qbs_only = tracking_filtered_by_window_length[tracking_filtered_by_window_length['pff_positionLinedUp'] == 'QB']
sum_outside_tacklebox = ((
    (qbs_only['y'] >= qbs_only[['y_LT', 'y_RT']].min(axis=1)) &
    (qbs_only['y'] <= qbs_only[['y_LT', 'y_RT']].max(axis=1))
) == False).sum()
print(sum_outside_tacklebox)


0
10
25
0


In [15]:
# tracking_filtered_by_window_length.to_csv("../outputs_csv/tracking_filtered_by_play_and_frame.csv", index=False)
tracking_filtered_by_window_length.isna().sum()

gameId                    0
playId                    0
season                    0
week                      0
gameDate                  0
                     ...   
event               3400144
frameIdBallSnap           0
y_LT                      0
y_RT                      0
frameIdEndWindow          0
Length: 65, dtype: int64

In [16]:
base_filtered = base_filtered.merge(
    play_windows[['gameId','playId']],
    on=['gameId','playId'],
    how='inner'
)

base_filtered = base_filtered.merge(
    tracking_filtered_by_window_length.drop_duplicates(['gameId','playId'])[['gameId','playId','frameIdEndWindow']], 
    on=['gameId','playId'],
    how='inner'
)

base_filtered = base_filtered[
    base_filtered['frameID'] <= base_filtered['frameIdEndWindow']
]

base_filtered

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.200,...,right,37.78,24.22,0.23,0.11,0.02,164.33,92.87,NaN,36
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.300,...,right,37.78,24.24,0.16,0.10,0.01,160.24,68.55,NaN,36
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.400,...,right,37.73,24.25,0.15,0.24,0.06,152.13,296.85,NaN,36
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.500,...,right,37.69,24.26,0.25,0.18,0.04,148.33,287.55,NaN,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6740224,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,right,40.36,44.25,7.47,0.28,0.74,48.35,60.83,NaN,37
6740225,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,right,41.02,44.60,7.45,0.67,0.75,48.35,61.74,NaN,37
6740226,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,right,41.68,44.94,7.48,1.43,0.75,58.99,63.79,NaN,37
6740227,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,right,42.36,45.25,7.41,2.01,0.75,63.66,65.69,NaN,37


In [17]:
base_filtered.drop_duplicates(subset=['gameId','playId'])

,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:31.100,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
946,2021090900,137,2021,1,09/09/2021,1,1,10,13:18,28:09.900,...,left,106.92,20.83,0.07,0.08,0.01,153.18,250.65,NaN,31
1760,2021090900,187,2021,1,09/09/2021,1,2,6,12:23,29:15.000,...,left,75.23,28.60,0.27,0.54,0.03,131.50,46.10,line_set,27
2442,2021090900,282,2021,1,09/09/2021,1,1,10,9:56,31:51.600,...,left,48.71,28.12,0.20,0.19,0.03,88.15,43.07,NaN,36
3388,2021090900,349,2021,1,09/09/2021,1,3,15,9:46,34:05.000,...,left,53.50,36.53,0.03,0.03,0.03,85.90,72.33,NaN,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6734618,2021110100,4310,2021,8,11/01/2021,4,3,8,1:56,15:52.300,...,left,18.90,47.67,0.00,0.00,0.00,241.60,341.47,NaN,37
6735938,2021110100,4363,2021,8,11/01/2021,4,1,10,1:07,18:40.400,...,right,33.29,26.85,0.00,0.00,0.00,86.74,34.36,NaN,37
6736994,2021110100,4392,2021,8,11/01/2021,4,2,7,1:01,19:17.300,...,right,36.31,20.84,0.00,0.00,0.00,89.16,353.34,NaN,37
6738160,2021110100,4411,2021,8,11/01/2021,4,3,15,0:39,19:39.600,...,right,27.97,20.56,0.01,0.01,0.01,92.81,250.80,NaN,32


In [18]:
# base_filtered.to_csv("../outputs_csv/base_filtered.csv", index=False)

In [19]:
filtered_df = tracking_filtered_by_window_length
filtered_df = filtered_df.drop(['y_LT','y_RT','ttt_le_1_5_and_behind_los'],axis=1)
pass_blockers = filtered_df[~filtered_df['pff_blockType'].isna()]
pass_rushers = filtered_df[filtered_df['pff_role'] == 'Pass Rush']

In [20]:
# calculate weighted distances

blockers = pass_blockers[
    ['gameId', 'playId', 'frameID', 'nflId', 'x', 'y', 'o']
].copy()

rushers = pass_rushers[
    ['gameId', 'playId', 'frameID', 'nflId', 'x', 'y']
].copy()

blockers = blockers.rename(columns={
    'nflId': 'blocker_nflId',
    'x': 'x_blocker',
    'y': 'y_blocker',
    'o': 'o_blocker'
})

rushers = rushers.rename(columns={
    'nflId': 'rusher_nflId',
    'x': 'x_rusher',
    'y': 'y_rusher'
})

# Every blocker paired with every rusher in the same frame
blocker_rusher_pairs = blockers.merge(
    rushers,
    on=['gameId', 'playId', 'frameID'],
    how='inner'
)

# Vector from blocker to rusher
blocker_rusher_pairs['dx'] = (
    blocker_rusher_pairs['x_rusher'] - blocker_rusher_pairs['x_blocker']
)
blocker_rusher_pairs['dy'] = (
    blocker_rusher_pairs['y_rusher'] - blocker_rusher_pairs['y_blocker']
)

# Euclidean distance
blocker_rusher_pairs['actual_distance'] = np.sqrt(
    blocker_rusher_pairs['dx']**2 + blocker_rusher_pairs['dy']**2
)

# Blocker facing direction unit vector under NFL angle convention:
# 0° = (0,1), 90° = (1,0)
o_rad = np.radians(blocker_rusher_pairs['o_blocker'])
blocker_rusher_pairs['ux_blocker'] = np.sin(o_rad)
blocker_rusher_pairs['uy_blocker'] = np.cos(o_rad)

# Unit vector from blocker to rusher
nonzero_dist = blocker_rusher_pairs['actual_distance'] > 0

blocker_rusher_pairs['ux_to_rusher'] = np.where(
    nonzero_dist,
    blocker_rusher_pairs['dx'] / blocker_rusher_pairs['actual_distance'],
    np.nan
)
blocker_rusher_pairs['uy_to_rusher'] = np.where(
    nonzero_dist,
    blocker_rusher_pairs['dy'] / blocker_rusher_pairs['actual_distance'],
    np.nan
)

# cos(theta) using dot product
blocker_rusher_pairs['cos_theta'] = (
    blocker_rusher_pairs['ux_blocker'] * blocker_rusher_pairs['ux_to_rusher']
    + blocker_rusher_pairs['uy_blocker'] * blocker_rusher_pairs['uy_to_rusher']
).clip(-1, 1)

# Optional: recover theta in degrees
blocker_rusher_pairs['theta_deg'] = np.degrees(
    np.arccos(blocker_rusher_pairs['cos_theta'])
)

# Weighted distance
blocker_rusher_pairs['weighted_distance'] = np.where(
    blocker_rusher_pairs['cos_theta'] > 0,
    blocker_rusher_pairs['actual_distance'] / blocker_rusher_pairs['cos_theta'],
    np.inf
)

# Optional final column selection
blocker_rusher_pairs = blocker_rusher_pairs[
    [
        'gameId', 'playId', 'frameID',
        'blocker_nflId', 'rusher_nflId',
        'x_blocker', 'y_blocker', 'o_blocker',
        'x_rusher', 'y_rusher',
        'dx', 'dy',
        'actual_distance', 'cos_theta', 'theta_deg', 'weighted_distance'
    ]
].copy()

blocker_rusher_pairs

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,dx,dy,actual_distance,cos_theta,theta_deg,weighted_distance
0,2021090900,97,11,40151,41263,41.58,24.31,71.03,42.34,19.20,0.76,-5.11,5.166208,-0.182416,100.510525,inf
1,2021090900,97,11,40151,42403,41.58,24.31,71.03,42.71,32.07,1.13,7.76,7.841843,0.457953,62.744905,17.123686
2,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,0.97,1.06,1.436837,0.878246,28.568545,1.636031
3,2021090900,97,11,40151,53441,41.58,24.31,71.03,42.81,22.24,1.23,-2.07,2.407862,0.203623,78.251096,11.825097
4,2021090900,97,11,40151,53504,41.58,24.31,71.03,43.25,26.78,1.67,2.47,2.981577,0.798984,36.966849,3.731712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742498,2021110100,4433,36,52507,52585,22.41,26.22,217.53,21.04,20.71,-1.37,-5.51,5.677764,0.916592,23.567181,6.194429
3742499,2021110100,4433,37,52507,42406,22.14,26.15,207.07,21.81,24.27,-0.33,-1.88,1.908743,0.955720,17.114186,1.997178
3742500,2021110100,4433,37,52507,43326,22.14,26.15,207.07,22.17,17.56,0.03,-8.59,8.590052,0.888856,27.270101,9.664162
3742501,2021110100,4433,37,52507,43338,22.14,26.15,207.07,21.73,25.86,-0.41,-0.29,0.502195,0.885738,27.657579,0.566980


In [21]:
blocker_rusher_pairs = blocker_rusher_pairs[
    (blocker_rusher_pairs['weighted_distance'] >= 0) &
    (blocker_rusher_pairs['weighted_distance'] <= 3.5)
].copy()

closest_rusher_per_blocker = (
    blocker_rusher_pairs
    .sort_values(
        ['gameId', 'playId', 'frameID', 'blocker_nflId', 'weighted_distance']
    )
    .drop_duplicates(
        subset=['gameId', 'playId', 'frameID', 'blocker_nflId'],
        keep='first'
    )
    .copy()
)

In [22]:
closest_rusher_per_blocker

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,dx,dy,actual_distance,cos_theta,theta_deg,weighted_distance
2,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,0.97,1.06,1.436837,0.878246,28.568545,1.636031
132,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,2.09,-1.93,2.844820,0.973382,13.249226,2.922613
262,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,0.96,-0.30,1.005783,0.988824,8.574025,1.017151
393,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,1.66,-0.46,1.722556,0.998252,3.388501,1.725573
523,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,2.21,1.26,2.543954,0.958078,16.649010,2.655268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3742497,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,-0.50,-0.16,0.524976,0.821892,34.725328,0.638741
3741878,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,0.29,-0.73,0.785493,0.999920,0.724041,0.785556
3742083,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,-0.23,0.50,0.550364,0.974222,13.037570,0.564926
3742396,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,0.08,-2.49,2.491285,0.979960,11.489805,2.542231


In [7]:
players = pd.read_csv("../cleaned_csv/players_cleaned.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")

In [9]:
players

,nflId,height,weight,birthDate,collegeName,officialPosition,displayName
0,25511,6-4,225,1977-08-03,Michigan,QB,Tom Brady
1,28963,6-5,240,1982-03-02,"Miami, O.",QB,Ben Roethlisberger
2,29550,6-4,328,1982-01-22,Arkansas,T,Jason Peters
3,29851,6-2,225,1983-12-02,California,QB,Aaron Rodgers
4,30078,6-2,228,1982-11-24,Harvard,QB,Ryan Fitzpatrick
...,...,...,...,...,...,...,...
1674,53991,6-1,320,NaN,NaN,DT,Forrest Merrill
1675,53994,6-5,300,NaN,NaN,C,Ryan McCollum
1676,53999,6-4,312,NaN,NaN,DT,Jack Heflin
1677,54006,6-6,330,NaN,NaN,T,Jake Curhan


In [ ]:
merged_df = closest_rusher_per_blocker.merge(
    filtered_df[['gameId', 'playId', 'frameID', 'nflId','quarter','gameClock']],
    left_on=['gameId', 'playId', 'frameID', 'blocker_nflId'],
    right_on=['gameId', 'playId', 'frameID', 'nflId'],
    how='left'
).drop(columns=['nflId'])


# Add blocker_name and rusher_name from players
name_map = players[['nflId', 'displayName']].drop_duplicates()

merged_df = merged_df.merge(
    name_map.rename(columns={
        'nflId': 'blocker_nflId',
        'displayName': 'blocker_name'
    }),
    on='blocker_nflId',
    how='left'
)

merged_df = merged_df.merge(
    name_map.rename(columns={
        'nflId': 'rusher_nflId',
        'displayName': 'rusher_name'
    }),
    on='rusher_nflId',
    how='left'
)

# Add blocker_position and rusher_position from players using officialPosition
player_positions = players[
    ['nflId', 'officialPosition']
].drop_duplicates(subset=['nflId'])

merged_df = merged_df.merge(
    player_positions.rename(columns={
        'nflId': 'blocker_nflId',
        'officialPosition': 'blocker_position'
    }),
    on='blocker_nflId',
    how='left'
)

merged_df = merged_df.merge(
    player_positions.rename(columns={
        'nflId': 'rusher_nflId',
        'officialPosition': 'rusher_position'
    }),
    on='rusher_nflId',
    how='left'
)

In [25]:
merged_df

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
0,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,...,1.436837,0.878246,28.568545,1.636031,1,13:33,Ryan Jensen,Carlos Watkins,C,DRT
1,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,...,2.844820,0.973382,13.249226,2.922613,1,13:33,Donovan Smith,Carlos Watkins,LT,DRT
2,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,...,1.005783,0.988824,8.574025,1.017151,1,13:33,Ali Marpet,Carlos Watkins,LG,DRT
3,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,...,1.722556,0.998252,3.388501,1.725573,1,13:33,Alex Cappa,Micah Parsons,RG,LILB
4,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,...,2.543954,0.958078,16.649010,2.655268,1,13:33,Tristan Wirfs,Micah Parsons,RT,LILB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775783,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,...,0.524976,0.821892,34.725328,0.638741,4,0:35,Matt Peart,Jarran Reed,LT,RE
775784,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,...,0.785493,0.999920,0.724041,0.785556,4,0:35,Nate Solder,Michael Danna,RT,LEO
775785,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,...,0.550364,0.974222,13.037570,0.564926,4,0:35,Matt Skura,Frank Clark,LG,ROLB
775786,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,...,2.491285,0.979960,11.489805,2.542231,4,0:35,Will Hernandez,Chris Jones,RG,LE


In [26]:
merged_df['rusher_position'].value_counts()

rusher_position
DRT      122107
DLT      111672
LE        85371
LEO       85144
REO       81175
ROLB      68078
LOLB      63486
RE        63233
NT        31989
NRT       18202
NLT       14912
RILB       8924
LILB       8908
MLB        3521
LLB        3061
RLB        3004
SCBL        961
SCBR        862
SCBiL       259
SCBoL       226
SCBiR       221
LCB         197
RCB         147
SCBoR       117
SSR          11
Name: count, dtype: int64

In [27]:
# pd.read_csv("../outputs_csv/blocker_rusher_matchups.csv")
# base_filtered.iloc[:,10:]
merged_df.drop_duplicates(subset=['gameId','playId'])

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
0,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,...,1.436837,0.878246,28.568545,1.636031,1,13:33,Ryan Jensen,Carlos Watkins,C,DRT
104,2021090900,137,12,37082,35454,109.64,20.43,192.24,108.11,19.67,...,1.708362,0.624630,51.344937,2.734996,1,13:18,Tyron Smith,Jason Pierre-Paul,LT,ROLB
190,2021090900,187,11,37082,35454,78.56,22.95,262.30,77.11,21.33,...,2.174144,0.760752,40.469494,2.857889,1,12:23,Tyron Smith,Jason Pierre-Paul,LT,REO
275,2021090900,282,11,42654,40074,50.22,32.23,253.11,48.69,32.03,...,1.543017,0.986451,9.442583,1.564211,1,9:56,La'el Collins,William Gholston,RT,LE
432,2021090900,349,12,42654,41915,56.71,33.07,272.53,54.07,33.58,...,2.688810,0.989263,8.403817,2.717994,1,9:46,La'el Collins,Shaquil Barrett,RT,LOLB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775272,2021110100,4310,12,43367,42349,19.18,28.14,269.92,17.41,27.89,...,1.787568,0.990366,7.959445,1.804957,4,1:56,Joe Thuney,Leonard Williams,LG,DRT
775369,2021110100,4363,12,37090,43326,32.73,26.57,101.92,34.66,26.90,...,1.958009,0.929630,21.622857,2.106225,4,1:07,Nate Solder,Chris Jones,RT,LE
775486,2021110100,4392,12,37090,43326,35.89,20.68,80.17,37.72,21.12,...,1.882153,0.997928,3.689414,1.886062,4,1:01,Nate Solder,Chris Jones,RT,LE
775571,2021110100,4411,12,37090,43326,27.46,20.39,104.76,29.54,20.87,...,2.134666,0.884950,27.754617,2.412188,4,0:39,Nate Solder,Chris Jones,RT,LE


In [29]:
merged_df[(merged_df['gameId'] == 2021093000) & (merged_df['playId'] == 621)].iloc[0:50]

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
309218,2021093000,621,11,41322,43455,63.25,27.67,89.22,64.91,27.63,...,1.660482,0.999289,2.160354,1.661663,1,3:25,Brandon Linder,D.J. Reader,C,NT
309219,2021093000,621,11,41939,48145,62.99,29.64,73.93,64.68,30.84,...,2.072704,0.943761,19.307006,2.196218,1,3:25,Andrew Norwell,Wyatt Ray,LG,RE
309220,2021093000,621,11,42410,43455,62.97,26.12,64.55,64.91,27.63,...,2.458394,0.976502,12.445425,2.517552,1,3:25,A.J. Cann,D.J. Reader,RG,NT
309221,2021093000,621,11,44846,48145,62.63,31.25,79.92,64.68,30.84,...,2.090598,0.931120,21.389932,2.245251,1,3:25,Cam Robinson,Wyatt Ray,LT,RE
309222,2021093000,621,11,47818,46138,62.20,24.60,98.50,64.66,25.28,...,2.552254,0.913886,23.952011,2.792749,1,3:25,Jawaan Taylor,B.J. Hill,RT,LE
309223,2021093000,621,12,41322,43455,63.16,27.64,93.16,64.72,27.57,...,1.561570,0.999947,0.590759,1.561653,1,3:25,Brandon Linder,D.J. Reader,C,NT
309224,2021093000,621,12,41939,48145,62.97,29.73,73.93,64.52,30.86,...,1.918176,0.939554,20.023256,2.041582,1,3:25,Andrew Norwell,Wyatt Ray,LG,RE
309225,2021093000,621,12,42410,43455,62.95,26.00,69.55,64.72,27.57,...,2.365967,0.932808,21.123207,2.536393,1,3:25,A.J. Cann,D.J. Reader,RG,NT
309226,2021093000,621,12,44846,48145,62.51,31.32,77.88,64.52,30.86,...,2.061965,0.906230,25.010481,2.275321,1,3:25,Cam Robinson,Wyatt Ray,LT,RE
309227,2021093000,621,12,47818,46138,62.02,24.55,98.50,64.48,25.22,...,2.549608,0.915414,23.735409,2.785196,1,3:25,Jawaan Taylor,B.J. Hill,RT,LE


In [39]:
merged_df.columns

Index(['gameId', 'playId', 'frameID', 'blocker_nflId', 'rusher_nflId',
       'x_blocker', 'y_blocker', 'o_blocker', 'x_rusher', 'y_rusher', 'dx',
       'dy', 'actual_distance', 'cos_theta', 'theta_deg', 'weighted_distance',
       'quarter', 'gameClock', 'blocker_name', 'rusher_name',
       'blocker_position', 'rusher_position'],
      dtype='str')

In [40]:
merged_df[merged_df['gameId'] == 2021093000].drop_duplicates(subset=['gameId','playId'])

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
308607,2021093000,169,11,41322,43455,55.52,29.86,81.63,57.02,29.91,...,1.500833,0.993649,6.460848,1.510426,1,12:28,Brandon Linder,D.J. Reader,C,NT
308737,2021093000,209,11,41322,44877,57.75,23.75,95.56,58.88,23.09,...,1.308625,0.908304,24.727953,1.440735,1,11:05,Brandon Linder,Larry Ogunjobi,C,DLT
308861,2021093000,329,11,38553,43352,72.18,27.21,251.66,70.27,26.50,...,2.037695,0.999359,2.051497,2.039002,1,9:28,Riley Reiff,Adam Gotsis,RT,LE
308937,2021093000,351,12,38553,44880,72.21,27.11,266.99,70.18,26.75,...,2.061674,0.992447,7.046276,2.077364,1,9:25,Riley Reiff,Dawuane Smoot,RT,LE
309081,2021093000,599,11,41322,43455,63.41,28.24,77.58,64.85,28.31,...,1.441700,0.985888,9.636979,1.462337,1,3:35,Brandon Linder,D.J. Reader,C,NT
309218,2021093000,621,11,41322,43455,63.25,27.67,89.22,64.91,27.63,...,1.660482,0.999289,2.160354,1.661663,1,3:25,Brandon Linder,D.J. Reader,C,NT
309348,2021093000,766,11,38553,44880,84.83,29.46,286.53,83.11,31.66,...,2.792562,0.814611,35.451057,3.428091,1,0:47,Riley Reiff,Dawuane Smoot,RT,LOLB
309459,2021093000,788,11,38553,43352,85.32,29.15,254.31,82.94,27.83,...,2.721544,0.973084,13.323708,2.796824,1,0:43,Riley Reiff,Adam Gotsis,RT,DLT
309592,2021093000,893,11,41939,45226,76.28,23.80,255.02,74.42,23.13,...,1.976993,0.996449,4.829784,1.984038,2,14:21,Andrew Norwell,Josh Tupou,LG,DRT
309707,2021093000,957,11,38553,43352,35.60,20.79,76.47,37.10,21.65,...,1.729046,0.959820,16.297085,1.801428,2,14:10,Riley Reiff,Adam Gotsis,RT,DLT


In [30]:



# base_filtered = base_filtered.merge(
# merged_df[['gameId', 'playId']].drop_duplicates(),
# on=['gameId', 'playId'],
# how='inner'
# )

# base_filtered = base_filtered.merge(
# filtered_df[['gameId', 'playId', 'frameIdBallSnap']].drop_duplicates(['gameId', 'playId']),
# on=['gameId', 'playId'],
# how='left'
# )
base_filtered = base_filtered.drop(['ttt_le_1_5_and_behind_los'],axis=1)
base_filtered=base_filtered.rename(columns={'frameID':'frameId'})


In [31]:
base_filtered.drop_duplicates(subset=['gameId','playId']).iloc[:,10:]

,frameId,possessionTeam,defensiveTeam,yardlineSide,yardlineNumber,absoluteYardlineNumber,offenseFormation,offenseRB,offenseTE,offenseWR,...,playDirection,x,y,s,a,dis,o,dir,event,frameIdEndWindow
0,1,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0,...,right,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36
946,1,DAL,TB,DAL,2,108,EMPTY,1.0,2.0,2.0,...,left,106.92,20.83,0.07,0.08,0.01,153.18,250.65,NaN,31
1760,1,DAL,TB,DAL,34,76,SHOTGUN,0.0,2.0,3.0,...,left,75.23,28.60,0.27,0.54,0.03,131.50,46.10,line_set,27
2442,1,DAL,TB,TB,39,49,SINGLEBACK,1.0,2.0,2.0,...,left,48.71,28.12,0.20,0.19,0.03,88.15,43.07,NaN,36
3388,1,DAL,TB,TB,44,54,SHOTGUN,1.0,1.0,3.0,...,left,53.50,36.53,0.03,0.03,0.03,85.90,72.33,NaN,32
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6734618,1,KC,NYG,NYG,8,18,SHOTGUN,1.0,1.0,3.0,...,left,18.90,47.67,0.00,0.00,0.00,241.60,341.47,NaN,37
6735938,1,NYG,KC,NYG,25,35,SHOTGUN,1.0,1.0,3.0,...,right,33.29,26.85,0.00,0.00,0.00,86.74,34.36,NaN,37
6736994,1,NYG,KC,NYG,28,38,SHOTGUN,1.0,1.0,3.0,...,right,36.31,20.84,0.00,0.00,0.00,89.16,353.34,NaN,37
6738160,1,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0,...,right,27.97,20.56,0.01,0.01,0.01,92.81,250.80,NaN,32


In [ ]:
blocker_rusher_output_df=merged_df.rename(columns={'frameID':'frameId'})
blocker_rusher_output_df.to_csv("../outputs_csv/blocker_rusher_matchups.csv", index=False)
base_filtered.to_csv("../outputs_csv/base_filtered.csv", index=False)


In [ ]:
# blocker_rusher_output_df.drop_duplicates(subset=['gameId','playId'])

,gameId,playId,frameId,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
0,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,...,1.436837,0.878246,28.568545,1.636031,1,13:33,Ryan Jensen,Carlos Watkins,C,DRT
104,2021090900,137,12,37082,35454,109.64,20.43,192.24,108.11,19.67,...,1.708362,0.624630,51.344937,2.734996,1,13:18,Tyron Smith,Jason Pierre-Paul,LT,ROLB
190,2021090900,187,11,37082,35454,78.56,22.95,262.30,77.11,21.33,...,2.174144,0.760752,40.469494,2.857889,1,12:23,Tyron Smith,Jason Pierre-Paul,LT,REO
275,2021090900,282,11,42654,40074,50.22,32.23,253.11,48.69,32.03,...,1.543017,0.986451,9.442583,1.564211,1,9:56,La'el Collins,William Gholston,RT,LE
432,2021090900,349,12,42654,41915,56.71,33.07,272.53,54.07,33.58,...,2.688810,0.989263,8.403817,2.717994,1,9:46,La'el Collins,Shaquil Barrett,RT,LOLB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775272,2021110100,4310,12,43367,42349,19.18,28.14,269.92,17.41,27.89,...,1.787568,0.990366,7.959445,1.804957,4,1:56,Joe Thuney,Leonard Williams,LG,DRT
775369,2021110100,4363,12,37090,43326,32.73,26.57,101.92,34.66,26.90,...,1.958009,0.929630,21.622857,2.106225,4,1:07,Nate Solder,Chris Jones,RT,LE
775486,2021110100,4392,12,37090,43326,35.89,20.68,80.17,37.72,21.12,...,1.882153,0.997928,3.689414,1.886062,4,1:01,Nate Solder,Chris Jones,RT,LE
775571,2021110100,4411,12,37090,43326,27.46,20.39,104.76,29.54,20.87,...,2.134666,0.884950,27.754617,2.412188,4,0:39,Nate Solder,Chris Jones,RT,LE


In [ ]:
# blocker_rusher_output_df.groupby(['gameId','playId','frameId','rusherId'])

,gameId,playId,frameId,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
0,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,...,1.436837,0.878246,28.568545,1.636031,1,13:33,Ryan Jensen,Carlos Watkins,C,DRT
1,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,...,2.844820,0.973382,13.249226,2.922613,1,13:33,Donovan Smith,Carlos Watkins,LT,DRT
2,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,...,1.005783,0.988824,8.574025,1.017151,1,13:33,Ali Marpet,Carlos Watkins,LG,DRT
3,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,...,1.722556,0.998252,3.388501,1.725573,1,13:33,Alex Cappa,Micah Parsons,RG,LILB
4,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,...,2.543954,0.958078,16.649010,2.655268,1,13:33,Tristan Wirfs,Micah Parsons,RT,LILB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775783,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,...,0.524976,0.821892,34.725328,0.638741,4,0:35,Matt Peart,Jarran Reed,LT,RE
775784,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,...,0.785493,0.999920,0.724041,0.785556,4,0:35,Nate Solder,Michael Danna,RT,LEO
775785,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,...,0.550364,0.974222,13.037570,0.564926,4,0:35,Matt Skura,Frank Clark,LG,ROLB
775786,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,...,2.491285,0.979960,11.489805,2.542231,4,0:35,Will Hernandez,Chris Jones,RG,LE


In [ ]:
# blocker_rusher_output_df = pd.read_csv("../outputs_csv/blocker_rusher_matchups.csv")
# base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")



In [ ]:
pass_rushers_output_df=pass_rushers.rename(columns={'frameID':'frameId'})
pass_rushers_output_df.to_csv("../outputs_csv/pass_rushers.csv", index=False)

In [ ]:
# pass_rushers = pd.read_csv("../outputs_csv/pass_rushers.csv")
# pass_rushers


/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_92456/1058389536.py:1: DtypeWarning: Columns (0: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  pass_rushers = pd.read_csv("../outputs_csv/pass_rushers.csv")


,gameId,playId,season,week,gameDate,quarter,down,yardsToGo,gameClock,play time,...,x,y,s,a,dis,o,dir,event,frameIdBallSnap,frameIdEndWindow
0,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.100,...,42.34,19.20,2.46,0.93,0.27,306.43,289.38,NaN,6,36
1,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.200,...,42.12,19.29,2.52,0.61,0.24,305.04,291.18,NaN,6,36
2,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.300,...,41.90,19.50,2.71,1.20,0.30,321.06,301.71,NaN,6,36
3,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.400,...,41.66,19.69,2.91,1.41,0.31,331.69,306.78,NaN,6,36
4,2021090900,97,2021,1,09/09/2021,1,3,2,13:33,26:32.500,...,41.42,19.90,3.05,1.37,0.32,343.10,310.65,NaN,6,36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
664542,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.400,...,21.30,20.27,1.97,2.51,0.20,323.88,307.18,NaN,7,37
664543,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.500,...,21.19,20.43,1.78,2.54,0.20,320.96,317.37,NaN,7,37
664544,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.600,...,21.09,20.56,1.58,2.49,0.16,318.47,325.26,NaN,7,37
664545,2021110100,4433,2021,8,11/01/2021,4,4,15,0:35,20:24.700,...,21.04,20.71,1.46,2.33,0.16,315.05,338.14,NaN,7,37


In [16]:
import pandas as pd

blocker_rusher_matchups = pd.read_csv("../outputs_csv/blocker_rusher_matchups.csv")
pass_rushers = pd.read_csv("../outputs_csv/pass_rushers.csv")

# --------------------------------------------------
# 1. Keep only the columns needed from pass_rushers
#    pass_rushers already contains only:
#    - relevant plays
#    - relevant frames
#    - players with pff_role == "Pass Rush"
# --------------------------------------------------
pass_rusher_frames = (
    pass_rushers[
        [
            "gameId",
            "playId",
            "frameId",
            "nflId",
            "displayName",
            "pff_positionLinedUp",
            "quarter",
            "gameClock"
        ]
    ]
    .drop_duplicates(subset=["gameId", "playId", "frameId", "nflId"])
    .rename(
        columns={
            "nflId": "rusher_nflId",
            "displayName": "rusher_name",
            "pff_positionLinedUp": "rusher_position"
        }
    )
    .copy()
)

# --------------------------------------------------
# 2. Compute frame-level attention score
#    Definition:
#    attention score at a frame = number of blocker rows
#    matched to that rusher at that [gameId, playId, frameId]
# --------------------------------------------------
frame_attention_counts = (
    blocker_rusher_matchups
    .groupby(["gameId", "playId", "frameId", "rusher_nflId"], as_index=False)
    .size()
    .rename(columns={"size": "attention_score_frame"})
)

# --------------------------------------------------
# 3. Merge frame-level counts onto the full pass_rusher frame set
#    This is important so that rushers with no matchup row on a
#    filtered frame still receive attention_score_frame = 0
# --------------------------------------------------
pass_rusher_attention_by_frame = pass_rusher_frames.merge(
    frame_attention_counts,
    on=["gameId", "playId", "frameId", "rusher_nflId"],
    how="left",
    validate="one_to_one"
)

pass_rusher_attention_by_frame["attention_score_frame"] = (
    pass_rusher_attention_by_frame["attention_score_frame"]
    .fillna(0)
    .astype(int)
)

# --------------------------------------------------
# 4. Aggregate to play-level average attention
#    Output: exactly one row per pass rusher per play
# --------------------------------------------------
pass_rusher_attention_by_play = (
    pass_rusher_attention_by_frame
    .groupby(
        ["gameId", "playId", "rusher_nflId", "rusher_name",'rusher_position'],
        as_index=False
    )
    .agg(
        attention_score_play=("attention_score_frame", "mean"),
        num_filtered_frames=("frameId", "nunique"),
        rusher_position=("rusher_position", "first")
    )
    .sort_values(["gameId", "playId", "rusher_nflId"])
    .reset_index(drop=True)
)

# Optional: make the play-level attention column name more explicit
pass_rusher_attention_by_play["attention_score_play"] = (
    pass_rusher_attention_by_play["attention_score_play"].astype(float)
)

# --------------------------------------------------
# 5. Display results
# --------------------------------------------------
print("pass_rusher_attention_by_frame shape:", pass_rusher_attention_by_frame.shape)
print("pass_rusher_attention_by_play shape:", pass_rusher_attention_by_play.shape)

pass_rusher_attention_by_frame.head(), pass_rusher_attention_by_play.head()

/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_97486/1480329881.py:4: DtypeWarning: Columns (0: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  pass_rushers = pd.read_csv("../outputs_csv/pass_rushers.csv")


pass_rusher_attention_by_frame shape: (664547, 9)
pass_rusher_attention_by_play shape: (30971, 7)


(       gameId  playId  frameId  rusher_nflId        rusher_name  \
 0  2021090900      97       11         41263  Demarcus Lawrence   
 1  2021090900      97       12         41263  Demarcus Lawrence   
 2  2021090900      97       13         41263  Demarcus Lawrence   
 3  2021090900      97       14         41263  Demarcus Lawrence   
 4  2021090900      97       15         41263  Demarcus Lawrence   
 
   rusher_position  quarter gameClock  attention_score_frame  
 0             LEO        1     13:33                      0  
 1             LEO        1     13:33                      1  
 2             LEO        1     13:33                      1  
 3             LEO        1     13:33                      1  
 4             LEO        1     13:33                      1  ,
        gameId  playId  rusher_nflId        rusher_name  attention_score_play  \
 0  2021090900      97         41263  Demarcus Lawrence              0.461538   
 1  2021090900      97         42403      Randy G

In [168]:
pass_rusher_attention_by_play[pass_rusher_attention_by_play['rusher_name'] == 'Aaron Donald'].iloc[0:50]

,gameId,playId,rusher_nflId,rusher_name,attention_score_play,num_filtered_frames,rusher_position
3631,2021091213,69,41239,Aaron Donald,1.000000,16,DLT
3635,2021091213,217,41239,Aaron Donald,1.789474,19,LE
3645,2021091213,377,41239,Aaron Donald,1.428571,14,RE
3649,2021091213,425,41239,Aaron Donald,2.000000,13,RE
3653,2021091213,468,41239,Aaron Donald,0.923077,13,REO
3658,2021091213,492,41239,Aaron Donald,1.800000,15,LE
3680,2021091213,775,41239,Aaron Donald,1.192308,26,RE
3684,2021091213,804,41239,Aaron Donald,1.523810,21,RE
3701,2021091213,1141,41239,Aaron Donald,1.333333,18,RE
3705,2021091213,1265,41239,Aaron Donald,1.076923,13,LE


In [17]:
# --------------------------------------------------
# Verification: check that there is exactly one row
# per pass rusher per play
# --------------------------------------------------
dup_counts = (
    pass_rusher_attention_by_play
    .groupby(["gameId", "playId", "rusher_nflId"])
    .size()
    .reset_index(name="row_count")
)

bad_keys = dup_counts[dup_counts["row_count"] != 1]

print("Total rows in play-level dataframe:", len(pass_rusher_attention_by_play))
print("Unique (gameId, playId, rusher_nflId) keys:", dup_counts.shape[0])
print("Number of bad keys:", bad_keys.shape[0])

if bad_keys.empty:
    print("Verified: exactly one row for each pass rusher for each play.")
else:
    print("Found keys with row_count != 1:")
    display(bad_keys.head(20))

Total rows in play-level dataframe: 30971
Unique (gameId, playId, rusher_nflId) keys: 30971
Number of bad keys: 0
Verified: exactly one row for each pass rusher for each play.


In [47]:
# --------------------------------------------------
# 6. Aggregate to player-level across all plays
#    This takes each rusher's average play-level attention score
#    across all plays in the filtered dataset
# --------------------------------------------------
pass_rusher_attention_overall = (
    pass_rusher_attention_by_play
    .groupby(["rusher_nflId", "rusher_name"], as_index=False)
    .agg(
        avg_attention_across_plays=("attention_score_play", "mean"),
        num_plays=("playId", "count")
    )
)

pass_rusher_attention_overall_50_plus = (
    pass_rusher_attention_overall[
        pass_rusher_attention_overall["num_plays"] >= 50
    ]
    .sort_values(
        ["avg_attention_across_plays", "num_plays"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

pass_rusher_attention_overall_50_plus

,rusher_nflId,rusher_name,avg_attention_across_plays,num_plays
0,53467,Christian Barmore,1.612486,165
1,37308,Lawrence Guy,1.594103,61
2,41239,Aaron Donald,1.590982,229
3,52455,Marlon Davidson,1.576065,50
4,46249,Folorunso Fatukasi,1.543350,102
...,...,...,...,...
219,48089,Malik Reed,0.803946,183
220,52492,Terrell Lewis,0.782586,136
221,52428,K'Lavon Chaisson,0.755497,50
222,47939,Justin Hollins,0.728201,64


In [49]:
pass_rusher_attention_overall_50_plus

,rusher_nflId,rusher_name,avg_attention_across_plays,num_plays
0,53467,Christian Barmore,1.612486,165
1,37308,Lawrence Guy,1.594103,61
2,41239,Aaron Donald,1.590982,229
3,52455,Marlon Davidson,1.576065,50
4,46249,Folorunso Fatukasi,1.543350,102
...,...,...,...,...
219,48089,Malik Reed,0.803946,183
220,52492,Terrell Lewis,0.782586,136
221,52428,K'Lavon Chaisson,0.755497,50
222,47939,Justin Hollins,0.728201,64


In [45]:
pass_rusher_attention_overall_50_plus[pass_rusher_attention_overall_50_plus['rusher_position'] == 'DT']

,rusher_nflId,rusher_name,avg_attention_across_plays,num_plays,rusher_position
0,53467,Christian Barmore,1.612486,165,DT
1,37308,Lawrence Guy,1.594103,61,DT
2,41239,Aaron Donald,1.590982,229,DT
3,52455,Marlon Davidson,1.576065,50,DT
4,46249,Folorunso Fatukasi,1.543350,102,DT
...,...,...,...,...,...
100,52448,Ross Blacklock,1.338270,63,DT
104,52835,Michael Hoecht,1.325558,56,DT
105,46307,Zach Sieler,1.319101,73,DT
111,45011,D.J. Jones,1.245436,72,DT


In [73]:
players = pd.read_csv("../cleaned_csv/players_cleaned.csv")

In [74]:
# Add blocker_position and rusher_position from players using officialPosition
player_positions = players[
    ['nflId', 'officialPosition']
].drop_duplicates(subset=['nflId'])



pass_rusher_attention_overall_50_plus = pass_rusher_attention_overall_50_plus.merge(
    player_positions.rename(columns={
        'nflId': 'rusher_nflId',
        'officialPosition': 'rusher_position'
    }),
    on='rusher_nflId',
    how='left'
)